In [14]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score


# CSV読み込み
df = pd.read_csv("stylometry_features.csv")

df = df.fillna(0)


y = df["label"]


# 全特徴量
all_features = [
    col for col in df.columns
    if col not in ["filename", "label"]
]


function_features = [
    "機能語率",
    "ひらがな率",
    "文長標準偏差",
    "平均文長",
    "最大文長",
    "句点頻度",
    "読点頻度",
    "助詞割合",
    "助動詞割合",
    "連体詞割合",
    "接続詞割合",
    "感動詞割合",
    
    "Hapax率_func",
    "Yules_K_func",

    "助詞_は",
    "助詞_に",
    "助詞_て",
    "助詞_と",
    "助詞_か",
    "助詞_が",
    "助詞_で",
    "助詞_や",
    "助詞_を",
    "助詞_の",
    "助詞_ほど",
    "助詞_まで",
    "助詞_も",
    "助詞_から",
    "助詞_たり",
    "助詞_など",
    "助詞_ながら",
    "助詞_だけ",
    "助詞_ば",
    "助詞_へ",
    "助詞_のみ",
    "助詞_しも",
    "助詞_しか",
    "助詞_すら",
    "助詞_より",
    "助詞_こそ",
    "助詞_つつ",
    "助詞_くらい",
    "助詞_かしら",
    "助詞_ずつ"
]

content_features = [
    "内容語率",
    "総単語数",
    "異なり語数",
    "TTR",
    "カタカナ率",
    "漢字率",
    "英数字率",

    "Hapax率_cont",
    "Yules_K_cont",

    "名詞割合",
    "動詞割合",
    "形容詞割合",
    "副詞割合",
    "形状詞割合",

    "感嘆符数",
    "疑問符数"
]


features_sets = {
    "All": all_features,
    "Function": function_features,
    "Content": content_features
}

importance_results = {}

for name, features in features_sets.items():

    print("\n========")
    print(name)
    print("========")


    X = df[features]


    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )


    rf = RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )


    rf.fit(
        X_train,
        y_train
    )

    importance = pd.DataFrame({
        "features": X.columns,
        "importance": rf.feature_importances_
    })
    importance_results[name] = importance
    
    y_pred = rf.predict(
        X_test
    )


    print(
        classification_report(
            y_test,
            y_pred,
            zero_division=0
        )
    )


    print(
        "Accuracy:",
        accuracy_score(
            y_test,
            y_pred
        )
    )


All
              precision    recall  f1-score   support

          AI       1.00      1.00      1.00         6
       Human       1.00      1.00      1.00         6

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12

Accuracy: 1.0

Function
              precision    recall  f1-score   support

          AI       1.00      1.00      1.00         6
       Human       1.00      1.00      1.00         6

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12

Accuracy: 1.0

Content
              precision    recall  f1-score   support

          AI       0.75      1.00      0.86         6
       Human       1.00      0.67      0.80         6

    accuracy                           0.83        12
   macro avg       0.88      0.83      0.83        12
weighted avg       0.8

In [15]:
from sklearn.model_selection import cross_val_score

for name, features in features_sets.items():
    
    X = df[features]
    # 5-fold交差検証
    scores = cross_val_score(
        rf,
        X,
        y,
        cv=5
    )
    
    print("-------")
    print(name)
    print("--------")
    print(scores)
    print("平均：", scores.mean())

-------
All
--------
[1.         1.         1.         1.         0.83333333]
平均： 0.9666666666666666
-------
Function
--------
[1.         1.         1.         1.         0.91666667]
平均： 0.9833333333333334
-------
Content
--------
[0.91666667 0.91666667 0.91666667 0.91666667 0.66666667]
平均： 0.8666666666666666


In [16]:
for name, importance in importance_results.items():
    
    print("--------")
    print(name)
    print("--------")
    
    print(
        importance.sort_values(
            "importance",
            ascending=False
        )
    )

--------
All
--------
        features  importance
35          助詞_や    0.203796
22          助詞割合    0.096176
21         形状詞割合    0.059637
16  Yules_K_cont    0.054393
25         連体詞割合    0.046848
..           ...         ...
48        助詞_ながら    0.000000
53         助詞_しも    0.000000
59        助詞_くらい    0.000000
60        助詞_かしら    0.000000
61         助詞_ずつ    0.000000

[62 rows x 2 columns]
--------
Function
--------
        features  importance
21          助詞_や    0.214706
7           助詞割合    0.100672
18          助詞_か    0.082862
9          連体詞割合    0.068159
2         文長標準偏差    0.059579
26          助詞_も    0.056797
10         接続詞割合    0.049789
6           読点頻度    0.039587
31         助詞_だけ    0.038506
19          助詞_が    0.027860
22          助詞_を    0.027338
27         助詞_から    0.023935
0           機能語率    0.022366
13  Yules_K_func    0.021633
23          助詞_の    0.016799
4           最大文長    0.013200
8          助動詞割合    0.013079
5           句点頻度    0.013005
3           平均文長    0.011461


In [21]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score


# CSV読み込み
train_df = pd.read_csv("stylometry_features_train.csv")
test_df = pd.read_csv("stylometry_features_test.csv")

train_df = train_df.fillna(0)
test_df = test_df.fillna(0)

# テストデータに存在しない特徴量を0で追加
for col in train_df.columns:
    if col not in test_df.columns:
        test_df[col] = 0

# 列順を学習データに合わせる
test_df = test_df[train_df.columns]


# 全特徴量
all_features = [
    col for col in df.columns
    if col not in ["filename", "label"]
]


function_features = [
    "機能語率",
    "ひらがな率",
    "文長標準偏差",
    "平均文長",
    "最大文長",
    "句点頻度",
    "読点頻度",
    "助詞割合",
    "助動詞割合",
    "連体詞割合",
    "接続詞割合",
    "感動詞割合",
    
    "Hapax率_func",
    "Yules_K_func",

    "助詞_は",
    "助詞_に",
    "助詞_て",
    "助詞_と",
    "助詞_か",
    "助詞_が",
    "助詞_で",
    "助詞_や",
    "助詞_を",
    "助詞_の",
    "助詞_ほど",
    "助詞_まで",
    "助詞_も",
    "助詞_から",
    "助詞_たり",
    "助詞_など",
    "助詞_ながら",
    "助詞_だけ",
    "助詞_ば",
    "助詞_へ",
    "助詞_のみ",
    "助詞_しも",
    "助詞_しか",
    "助詞_すら",
    "助詞_より",
    "助詞_こそ",
    "助詞_つつ",
    "助詞_くらい",
    "助詞_かしら",
    "助詞_ずつ"
]

content_features = [
    "内容語率",
    "総単語数",
    "異なり語数",
    "TTR",
    "カタカナ率",
    "漢字率",
    "英数字率",

    "Hapax率_cont",
    "Yules_K_cont",

    "名詞割合",
    "動詞割合",
    "形容詞割合",
    "副詞割合",
    "形状詞割合",

    "感嘆符数",
    "疑問符数"
]


features_sets = {
    "All": all_features,
    "Function": function_features,
    "Content": content_features
}

importance_results = {}

for name, features in features_sets.items():

    print("\n========")
    print(name)
    print("========")
    
    X_train = train_df[features]
    y_train = train_df["label"]

    X_test = test_df[features]
    y_test = test_df["label"]


    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        random_state=42
    )


    rf.fit(
        X_train,
        y_train
    )

    importance = pd.DataFrame({
    "features": features,
    "importance": rf.feature_importances_
    })
    importance_results[name] = importance
    
    y_pred = rf.predict(
        X_test
    )


    print(
        classification_report(
            y_test,
            y_pred,
            zero_division=0
        )
    )


    print(
        "Accuracy:",
        accuracy_score(
            y_test,
            y_pred
        )
    )


All
              precision    recall  f1-score   support

          AI       0.90      0.90      0.90        10
       Human       0.90      0.90      0.90        10

    accuracy                           0.90        20
   macro avg       0.90      0.90      0.90        20
weighted avg       0.90      0.90      0.90        20

Accuracy: 0.9

Function
              precision    recall  f1-score   support

          AI       1.00      0.90      0.95        10
       Human       0.91      1.00      0.95        10

    accuracy                           0.95        20
   macro avg       0.95      0.95      0.95        20
weighted avg       0.95      0.95      0.95        20

Accuracy: 0.95

Content
              precision    recall  f1-score   support

          AI       0.80      0.80      0.80        10
       Human       0.80      0.80      0.80        10

    accuracy                           0.80        20
   macro avg       0.80      0.80      0.80        20
weighted avg       0.

In [24]:
for name, importance in importance_results.items():

    importance.sort_values(
        "importance",
        ascending=False
    ).to_csv(
        f"importance_{name}.csv",
        index=False,
        encoding="utf-8-sig"
    )